<a href="https://colab.research.google.com/github/teguhsmlnna666/BigData26_B_2411532005_TeguhEsaMaulanna/blob/main/Praktikum3/BD_P03_2411532005_TeguhEsaMaulanna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Langkah Praktikum

Langkah ini untuk menyiapkan lingkungan kerja Google Colab, memeriksa versi library yang digunakan, menghubungkan Google Drive, membuat direktori penyimpanan data, serta menyiapkan fungsi pengukuran waktu, penggunaan memori, dan pencatatan data lineage sebagai asal-usul data selama proses praktikum.

In [ ]:
import sys, time, os, json, sqlite3, gc
import pandas as pd, numpy as np, psutil
for nama in ["pandas", "numpy", "pyarrow", "requests", "sklearn", "sqlalchemy"]:
    try:
        mod = __import__(nama)
        print(f"{nama:11s} : {mod.__version__}")
    except ImportError:
        print(f"{nama:11s} : BELUM TERPASANG")
from google.colab import drive
drive.mount("/content/drive")
DIR_MENTAH = "/content/lapisan_mentah"
DIR_KURASI = "/content/lapisan_terkurasi"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum3"
for d in (DIR_MENTAH, DIR_KURASI, DIR_SIMPAN):
    os.makedirs(d, exist_ok=True)

# Dipakai kembali dari Praktikum 1 / K-3
proses = psutil.Process(os.getpid())
catatan = []
def rss_mb():
    return proses.memory_info().rss / 1024**2
def ukur(label, fungsi):
    m0, t0 = rss_mb(), time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0
    catatan.append({"langkah": label, "detik": round(detik, 2),
                    "delta_rss_mb": round(rss_mb() - m0, 1)})
    print(f"[{label}] {detik:.2f} s")
    return hasil

# Baru di Praktikum 3: catatan asal-usul data (lineage)
lineage = []
def catat_sumber(nama, asal, jumlah_baris, keterangan=""):
    lineage.append({
        "sumber": nama, "asal": asal,
        "diambil_pada": pd.Timestamp.now("UTC").isoformat(),
        "jumlah_baris": jumlah_baris, "keterangan": keterangan,
    })
    print(f"[lineage] {nama}: {jumlah_baris:,} baris")

pandas      : 2.2.3
numpy       : 2.1.3
pyarrow     : 23.0.1
requests    : 2.32.4
sklearn     : 1.6.1
sqlalchemy  : 2.0.54
Mounted at /content/drive


Langkah ini untuk mengunduh data perjalanan dan zona taksi secara aman dengan mekanisme retry, validasi file, dan penulisan atomik, kemudian membaca kolom yang diperlukan serta mencatat sumber dan jumlah baris data melalui data lineage.

In [ ]:
import requests, shutil, os, time

def unduh_aman(url, tujuan, percobaan=3, jeda_awal=2):
    """Unduh dengan retry + penulisan atomik. Melewati unduhan bila file sudah ada."""
    if os.path.exists(tujuan) and os.path.getsize(tujuan) > 0:
        print("cache ditemukan:", os.path.basename(tujuan))
        return tujuan

    sementara = tujuan + ".part"

    # Menambahkan identitas browser agar tidak diblokir server
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

    for i in range(percobaan):
        try:
            with requests.get(url, stream=True, timeout=60, headers=headers) as r:
                r.raise_for_status()
                with open(sementara, "wb") as f:
                    shutil.copyfileobj(r.raw, f)

            # Validasi jika file sangat kecil (misal kurang dari 1KB) kemungkinan error/kosong
            if os.path.getsize(sementara) < 1000:
                raise IOError(f"berkas terlalu kecil atau kosong (ukuran: {os.path.getsize(sementara)} bytes)")

            os.replace(sementara, tujuan) # atomik
            return tujuan
        except Exception as e:
            jeda = jeda_awal * (2 ** i) # exponential backoff
            print(f"percobaan {i+1} gagal ({e}); menunggu {jeda}s")
            time.sleep(jeda)

    raise RuntimeError(f"gagal mengunduh setelah {percobaan} percobaan: {url}")

URL_TRIP = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
URL_ZONA = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

PATH_TRIP = os.path.join(DIR_MENTAH, "yellow_tripdata_2023-01.parquet")
PATH_ZONA = os.path.join(DIR_MENTAH, "taxi_zone_lookup.csv")

ukur("unduh trip", lambda: unduh_aman(URL_TRIP, PATH_TRIP))
ukur("unduh zona", lambda: unduh_aman(URL_ZONA, PATH_ZONA))

for pth in (PATH_TRIP, PATH_ZONA):
    print(os.path.basename(pth), "->", round(os.path.getsize(pth)/1024**2, 2), "MB")

KOLOM = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count",
    "trip_distance", "PULocationID", "DOLocationID", "payment_type",
    "fare_amount", "tip_amount", "total_amount"
]

trip = ukur("baca trip", lambda: pd.read_parquet(PATH_TRIP, columns=KOLOM))
zona = pd.read_csv(URL_ZONA)

catat_sumber("trip", URL_TRIP, len(trip), "Parquet bulanan TLC")
catat_sumber("zona", URL_ZONA, len(zona), "tabel dimensi 265 zona")

display(zona.head())

[unduh trip] 0.34 s
[unduh zona] 0.05 s
yellow_tripdata_2023-01.parquet -> 45.46 MB
taxi_zone_lookup.csv -> 0.0 MB
[baca trip] 0.74 s
[lineage] trip: 3,066,766 baris
[lineage] zona: 265 baris


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


Langkah ini untuk mengambil data cuaca per jam melalui API Open-Meteo, melakukan validasi respons dan retry saat terjadi gangguan, kemudian mengubah format waktu serta nama kolom agar siap digunakan dalam proses analisis.

In [ ]:
API_CUACA = "https://archive-api.open-meteo.com/v1/archive"
def ambil_cuaca(mulai, selesai, lat=40.7128, lon=-74.0060, percobaan=3):
    parameter = {
        "latitude": lat, "longitude": lon,
        "start_date": mulai, "end_date": selesai,
        "hourly": "temperature_2m,precipitation",
        "timezone": "America/New_York",
    }
    for i in range(percobaan):
        r = requests.get(API_CUACA, params=parameter, timeout=60)
        if r.status_code == 200:
            data = r.json()
            if "hourly" not in data:  # validasi struktur
                raise ValueError("kunci 'hourly' tidak ada pada respons")
            return pd.DataFrame(data["hourly"])
        if r.status_code in (429, 500, 502, 503):  # layak dicoba ulang
            time.sleep(2**i)
            continue
        r.raise_for_status()  # galat lain: hentikan
    raise RuntimeError("API cuaca tidak merespons dengan benar")

cuaca = ukur("ambil cuaca", lambda: ambil_cuaca("2023-01-01", "2023-01-31"))
cuaca["time"] = pd.to_datetime(cuaca["time"])
cuaca = cuaca.rename(columns={"time": "jam_mulai",
                              "temperature_2m": "suhu_c",
                              "precipitation": "hujan_mm"})
catat_sumber("cuaca", API_CUACA, len(cuaca), "Open-Meteo hourly archive")
cuaca.head()

[ambil cuaca] 0.49 s
[lineage] cuaca: 744 baris


,jam_mulai,suhu_c,hujan_mm
0,2023-01-01 00:00:00,10.9,1.0
1,2023-01-01 01:00:00,10.6,1.0
2,2023-01-01 02:00:00,10.6,0.1
3,2023-01-01 03:00:00,10.5,0.0
4,2023-01-01 04:00:00,9.8,0.0


Langkah ini untuk membuat database SQLite sebagai sumber data operasional, menyimpan tabel referensi metode pembayaran dan data transaksi perjalanan, serta melakukan agregasi dan pembacaan data secara bertahap untuk mengurangi penggunaan memori.

In [ ]:
from sqlalchemy import create_engine
PATH_DB = os.path.join(DIR_MENTAH, "operasional.db")
engine = create_engine(f"sqlite:///{PATH_DB}")

tarif_referensi = pd.DataFrame({
    "payment_type": [1, 2, 3, 4, 5, 6],
    "nama_pembayaran": ["Kartu kredit", "Tunai", "Gratis",
                        "Sengketa", "Tidak diketahui", "Perjalanan batal"],
    "kena_biaya_admin": [1, 0, 0, 0, 0, 0],
})
tarif_referensi.to_sql("tarif_referensi", engine, if_exists="replace", index=False)

# tabel transaksi tiruan untuk latihan pembacaan bertahap
trip.head(300_000).to_sql(
    "trip_operasional", engine, if_exists="replace",
    index=False, chunksize=50_000)

print("Tabel dibuat:", pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'", engine)["name"].tolist())

SQL = """
SELECT payment_type, COUNT(*) AS jumlah, AVG(total_amount) AS rata_total
FROM trip_operasional
WHERE total_amount > 0
GROUP BY payment_type
ORDER BY jumlah DESC
"""
ringkas_db = pd.read_sql(SQL, engine)
print(ringkas_db)

# Pembacaan bertahap: agregasi tanpa memuat seluruh tabel ke memori
total_baris = 0
for bagian in pd.read_sql("SELECT trip_distance FROM trip_operasional",
                          engine, chunksize=50_000):
    total_baris += len(bagian)
print("dibaca bertahap:", f"{total_baris:,}", "baris")

tarif_referensi = pd.read_sql("SELECT * FROM tarif_referensi", engine)
catat_sumber("tarif_referensi", PATH_DB, len(tarif_referensi),
             "tabel dimensi dari basis data operasional")

Tabel dibuat: ['tarif_referensi', 'trip_operasional']
   payment_type  jumlah  rata_total
0             1  226967   31.339172
1             2   66504   25.218836
2             4    2082   26.611114
3             3    1553   22.089691
dibaca bertahap: 300,000 baris
[lineage] tarif_referensi: 6 baris


Langkah ini untuk membuat profil data, menghitung durasi perjalanan, serta memeriksa kualitas data berdasarkan kelengkapan, keunikan, validitas, konsistensi, dan ketepatan waktu untuk mengidentifikasi data yang tidak memenuhi aturan.

In [ ]:
trip["durasi_menit"] = (
    (trip["tpep_dropoff_datetime"] - trip["tpep_pickup_datetime"])
    .dt.total_seconds() / 60)

def profil_kolom(df):
    baris = []
    for kol in df.columns:
        s = df[kol]
        baris.append({
            "kolom": kol, "tipe": str(s.dtype),
            "persen_hilang": round(100 * s.isna().mean(), 3),
            "nilai_unik": s.nunique(dropna=True),
            "contoh": s.dropna().iloc[0] if s.notna().any() else None,
        })
    return pd.DataFrame(baris)

profil = profil_kolom(trip)
profil

AWAL, AKHIR = pd.Timestamp("2023-01-01"), pd.Timestamp("2023-02-01")
zona_sah = set(zona["LocationID"])
aturan = {
    "kelengkapan: passenger_count ada": trip["passenger_count"].notna(),
    "keunikan: baris tidak duplikat": ~trip.duplicated(),
    "validitas: jarak 0-100 mil": trip["trip_distance"].between(0.01, 100),
    "validitas: total_amount > 0": trip["total_amount"] > 0,
    "validitas: PULocationID dikenal": trip["PULocationID"].isin(zona_sah),
    "konsistensi: durasi 1-180 menit": trip["durasi_menit"].between(1, 180),
    "konsistensi: total >= fare": trip["total_amount"] >= trip["fare_amount"],
    "ketepatan waktu dalam Jan 2023": trip["tpep_pickup_datetime"].between(AWAL, AKHIR),
}

laporan = pd.DataFrame([
    {"aturan": nama, "lulus": int(mask.sum()), "gagal": int((~mask).sum()),
     "persen_gagal": round(100 * (~mask).mean(), 3)}
    for nama, mask in aturan.items()
]).sort_values("persen_gagal", ascending=False)
laporan

,aturan,lulus,gagal,persen_gagal
0,kelengkapan: passenger_count ada,2995023,71743,2.339
2,validitas: jarak 0-100 mil,3020816,45950,1.498
5,konsistensi: durasi 1-180 menit,3030383,36383,1.186
3,validitas: total_amount > 0,3040994,25772,0.840
6,konsistensi: total >= fare,3041743,25023,0.816
7,ketepatan waktu dalam Jan 2023,3066718,48,0.002
1,keunikan: baris tidak duplikat,3066766,0,0.000
4,validitas: PULocationID dikenal,3066766,0,0.000


Langkah ini untuk menangani missing value pada passenger_count dengan membandingkan beberapa strategi, kemudian memilih pengisian median berdasarkan jam perjalanan serta menghapus data duplikat berdasarkan kunci transaksi.

In [ ]:
KOL = "passenger_count"
asli = trip[KOL]
print("persen hilang:", round(100 * asli.isna().mean(), 3))

trip["jam"] = trip["tpep_pickup_datetime"].dt.hour
strategi = {
    "1_dibuang": asli.dropna(),
    "2_isi_median": asli.fillna(asli.median()),
    "3_isi_modus": asli.fillna(asli.mode().iloc[0]),
    "4_median_perjam": asli.fillna(trip.groupby("jam")[KOL].transform("median")),
}
banding = pd.DataFrame([
    {"strategi": nama, "n": len(s), "mean": round(s.mean(), 4),
     "median": s.median(), "std": round(s.std(), 4)}
    for nama, s in strategi.items()
])
banding

# Strategi yang dipilih modul ini: tandai + isi per kelompok (lihat G.4)
trip[KOL + "_hilang"] = trip[KOL].isna().astype("int8")
trip[KOL] = trip[KOL].fillna(trip.groupby("jam")[KOL].transform("median"))
trip[KOL] = trip[KOL].fillna(trip[KOL].median())  # jaring pengaman

# Duplikat: periksa dua tingkat
dup_penuh = trip.duplicated().sum()
KUNCI = ["tpep_pickup_datetime", "tpep_dropoff_datetime",
         "PULocationID", "DOLocationID", "total_amount"]
dup_kunci = trip.duplicated(subset=KUNCI).sum()
print(f"duplikat penuh: {dup_penuh:,} | duplikat pada kunci: {dup_kunci:,}")

sebelum = len(trip)
trip = trip.drop_duplicates(subset=KUNCI, keep="first").reset_index(drop=True)
print(f"{sebelum:,} -> {len(trip):,} baris")

persen hilang: 2.339
duplikat penuh: 0 | duplikat pada kunci: 1
3,066,766 -> 3,066,765 baris


Langkah ini untuk mendeteksi dan menangani outlier menggunakan metode IQR dan Z-score, menyaring data yang tidak layak, serta melakukan join dengan tabel zona, metode pembayaran, dan data cuaca untuk melengkapi informasi perjalanan.

In [ ]:
def batas_iqr(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

def ringkas_outlier(df, kolom):
    hasil = []
    for kol in kolom:
        s = df[kol].dropna()
        lo, hi = batas_iqr(s)
        z = (s - s.mean()) / s.std()
        hasil.append({
            "kolom": kol, "batas_bawah": round(lo, 2), "batas_atas": round(hi, 2),
            "outlier_iqr": int(((s < lo) | (s > hi)).sum()),
            "outlier_z3": int((z.abs() > 3).sum()), "p99": round(s.quantile(0.99), 2),
        })
    return pd.DataFrame(hasil)

ringkas_outlier(trip, ["trip_distance", "durasi_menit", "total_amount"])

# Tindakan berbeda untuk maksud berbeda (lihat G.5)
layak = (
    trip["trip_distance"].between(0.01, 100)
    & trip["durasi_menit"].between(1, 180)
    & (trip["total_amount"] > 0)
)
bersih = trip.loc[layak].copy()

# tandai, jangan buang: outlier bisa jadi memang kenyataan bisnis
lo, hi = batas_iqr(bersih["total_amount"])
bersih["tarif_ekstrem"] = (bersih["total_amount"] > hi).astype("int8")

# winsorizing hanya untuk kolom turunan yang akan dipakai model
batas_atas = bersih["trip_distance"].quantile(0.995)
bersih["trip_distance_capped"] = bersih["trip_distance"].clip(upper=batas_atas)

print(f"{len(trip):,} -> {len(bersih):,} baris "
      f"({100*(1-len(bersih)/len(trip)):.2f}% dibuang)")

# Join 1: tabel zona (dimensi)
print("kunci zona unik?", zona["LocationID"].is_unique)  # WAJIB diperiksa
n0 = len(bersih)
bersih = bersih.merge(
    zona[["LocationID", "Borough", "Zone"]]
    .rename(columns={"Borough": "borough_naik", "Zone": "zona_naik"}),
    left_on="PULocationID", right_on="LocationID",
    how="left", validate="many_to_one"
)
bersih = bersih.drop(columns="LocationID")  # gagal keras bila tidak 1:1
print(f"baris {n0:,} -> {len(bersih):,} (harus sama)")
print("zona tak dikenal:", bersih["zona_naik"].isna().sum())

# Join 2: tabel pembayaran dari basis data
bersih = bersih.merge(tarif_referensi[["payment_type", "nama_pembayaran"]],
                      on="payment_type", how="left", validate="many_to_one")

# Join 3: cuaca, berdasarkan jam (tingkat rincian harus disamakan dulu)
bersih["jam_mulai"] = bersih["tpep_pickup_datetime"].dt.floor("h")
n0 = len(bersih)
bersih = bersih.merge(cuaca, on="jam_mulai", how="left", validate="many_to_one")
print(f"baris {n0:,} -> {len(bersih):,}")
print("tanpa data cuaca:", bersih["suhu_c"].isna().sum())

3,066,765 -> 2,986,910 baris (2.60% dibuang)
kunci zona unik? True
baris 2,986,910 -> 2,986,910 (harus sama)
zona tak dikenal: 37609
baris 2,986,910 -> 2,986,910
tanpa data cuaca: 37


Langkah ini untuk membuat fitur turunan perjalanan dan melakukan prapemrosesan data dengan StandardScaler untuk fitur numerik serta OneHotEncoder untuk fitur kategorikal, dengan proses fitting hanya pada data latih untuk mencegah kebocoran data.

In [ ]:
bersih["hari_minggu"] = bersih["tpep_pickup_datetime"].dt.dayofweek
bersih["akhir_pekan"] = (bersih["hari_minggu"] >= 5).astype("int8")
bersih["kecepatan_mph"] = (bersih["trip_distance"] / (bersih["durasi_menit"] / 60)).round(2)
bersih["tarif_per_mil"] = (bersih["total_amount"] / bersih["trip_distance"]).round(3)
bersih["hujan"] = (bersih["hujan_mm"].fillna(0) > 0.1).astype("int8")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

FITUR_NUM = ["trip_distance_capped", "durasi_menit", "suhu_c"]
FITUR_KAT = ["nama_pembayaran", "borough_naik"]

contoh = bersih.dropna(subset=FITUR_NUM + FITUR_KAT).sample(
    n=min(200_000, len(bersih)), random_state=42)

latih, uji = train_test_split(contoh, test_size=0.2, random_state=42)

skala = StandardScaler().fit(latih[FITUR_NUM])  # fit HANYA di data latih
latih_num = skala.transform(latih[FITUR_NUM])
uji_num = skala.transform(uji[FITUR_NUM])       # transform saja, tanpa fit ulang

print("mean latih (harus ~0):", latih_num.mean(axis=0).round(3))
print("mean uji (tidak harus 0):", uji_num.mean(axis=0).round(3))

# Bergantung versi: parameter sparse_output ada sejak scikit-learn 1.2
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
enc.fit(latih[FITUR_KAT])
latih_kat = enc.transform(latih[FITUR_KAT])

print("jumlah kolom hasil one-hot:", latih_kat.shape[1])
print("nama kolom:", enc.get_feature_names_out()[:6], "...")

mean latih (harus ~0): [-0.  0. -0.]
mean uji (tidak harus 0): [ 0.001  0.001 -0.009]
jumlah kolom hasil one-hot: 11
nama kolom: ['nama_pembayaran_Gratis' 'nama_pembayaran_Kartu kredit'
 'nama_pembayaran_Sengketa' 'nama_pembayaran_Tunai' 'borough_naik_Bronx'
 'borough_naik_Brooklyn'] ...


Langkah ini untuk membangun pipeline pengolahan data yang idempoten, melakukan validasi struktur dan kualitas data, menyimpan hasil dalam format Parquet berpartisi, serta menghasilkan artefak berupa laporan kualitas, data lineage, pengukuran kinerja, dan arsip data bersih.

In [ ]:
KONTRAK = {
    "tpep_pickup_datetime": "datetime64[ns]", "trip_distance": "float64",
    "durasi_menit": "float64", "total_amount": "float64",
    "borough_naik": "object", "nama_pembayaran": "object",
}

def validasi(df, kontrak=KONTRAK):
    masalah = []
    for kol, tipe in kontrak.items():
        if kol not in df.columns:
            masalah.append(f"kolom hilang: {kol}")
        elif not str(df[kol].dtype).startswith(tipe.split("[")[0]):
            masalah.append(f"tipe {kol}: {df[kol].dtype} != {tipe}")
    if df.empty:
        masalah.append("DataFrame kosong")
    if len(df) != len(df.drop_duplicates(subset=KUNCI)):
        masalah.append("masih ada duplikat pada kunci")
    if masalah:
        raise AssertionError("Validasi gagal:\n- " + "\n- ".join(masalah))
    print(f"Validasi lulus: {len(df):,} baris x {df.shape[1]} kolom")
    return df

def pipeline(path_trip, sumber_zona, df_cuaca, df_tarif):
    """Satu fungsi, idempoten: input mentah -> DataFrame tervalidasi."""
    df = pd.read_parquet(path_trip, columns=KOLOM)

    # Penyesuaian K-2: membaca zona (bisa dari URL_ZONA atau PATH_ZONA)
    if os.path.exists(sumber_zona) and os.path.getsize(sumber_zona) > 1000:
        z = pd.read_csv(sumber_zona)
    else:
        z = pd.read_csv(URL_ZONA)

    df["durasi_menit"] = ((df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"])
                          .dt.total_seconds() / 60)
    df["jam"] = df["tpep_pickup_datetime"].dt.hour
    df["passenger_count"] = df["passenger_count"].fillna(
        df.groupby("jam")["passenger_count"].transform("median"))
    df = df.drop_duplicates(subset=KUNCI)
    df = df[
        df["trip_distance"].between(0.01, 100)
        & df["durasi_menit"].between(1, 180)
        & (df["total_amount"] > 0)
    ]
    df = (
        df.merge(z[["LocationID", "Borough"]]
          .rename(columns={"Borough": "borough_naik"}),
          left_on="PULocationID", right_on="LocationID",
          how="left", validate="many_to_one")
        .drop(columns="LocationID")
        .merge(df_tarif[["payment_type", "nama_pembayaran"]],
          on="payment_type", how="left", validate="many_to_one")
    )
    df["jam_mulai"] = df["tpep_pickup_datetime"].dt.floor("h")
    df = df.merge(df_cuaca, on="jam_mulai", how="left", validate="many_to_one")
    return validasi(df)

# Eksekusi pipeline menggunakan URL_ZONA atau PATH_ZONA
hasil = ukur("pipeline penuh", lambda: pipeline(PATH_TRIP, URL_ZONA, cuaca, tarif_referensi))

OUT = os.path.join(DIR_KURASI, "trips_bersih")
hasil["tanggal"] = hasil["tpep_pickup_datetime"].dt.date.astype(str)
ukur("tulis parquet berpartisi", lambda: hasil.to_parquet(
    OUT, partition_cols=["borough_naik"], index=False))
print("partisi yang terbentuk:", sorted(os.listdir(OUT))[:5], "...")

# Bukti idempotensi: jalankan ulang, hasil harus identik
ulang = pipeline(PATH_TRIP, URL_ZONA, cuaca, tarif_referensi)
print("idempoten?", len(ulang) == len(hasil))

# Artefak yang dikumpulkan
laporan.to_csv(f"{DIR_SIMPAN}/laporan_kualitas.csv", index=False)
pd.DataFrame(lineage).to_csv(f"{DIR_SIMPAN}/lineage.csv", index=False)
pd.DataFrame(catatan).to_csv(f"{DIR_SIMPAN}/pengukuran_kinerja.csv", index=False)
shutil.make_archive(f"{DIR_SIMPAN}/trips_bersih", "zip", OUT)

Validasi lulus: 2,986,910 baris x 17 kolom
[pipeline penuh] 5.77 s
[tulis parquet berpartisi] 2.04 s
partisi yang terbentuk: ['borough_naik=Bronx', 'borough_naik=Brooklyn', 'borough_naik=EWR', 'borough_naik=Manhattan', 'borough_naik=Queens'] ...
Validasi lulus: 2,986,910 baris x 17 kolom
idempoten? True


'/content/drive/MyDrive/BigData/Praktikum3/trips_bersih.zip'

Langkah ini untuk mengolah data perjalanan menggunakan Apache Spark, melakukan penyaringan dan penghapusan duplikat, menggabungkan data zona dengan broadcast join, menghitung jumlah data bersih, serta menyimpan hasil dalam Parquet yang dipartisi berdasarkan wilayah.

In [ ]:
!pip install -q pyspark==3.5.1
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName("BD-P02").master("local[*]")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

sdf = spark.read.parquet(PATH_TRIP).select(*KOLOM)
szona = spark.createDataFrame(zona[["LocationID", "Borough"]])

sbersih = (
    sdf
    .withColumn("durasi_menit",
                (F.unix_timestamp("tpep_dropoff_datetime")
                 - F.unix_timestamp("tpep_pickup_datetime")) / 60)
    .filter(F.col("trip_distance").between(0.01, 100))
    .filter(F.col("durasi_menit").between(1, 180))
    .filter(F.col("total_amount") > 0)
    .dropDuplicates(["tpep_pickup_datetime", "tpep_dropoff_datetime",
                     "PULocationID", "DOLocationID", "total_amount"])
    .join(F.broadcast(szona),  # tabel kecil: broadcast
          sdf.PULocationID == szona.LocationID, "left")
    .drop("LocationID")
)

ukur("spark: hitung baris bersih", lambda: sbersih.count())
sbersih.groupBy("Borough").count().orderBy(F.desc("count")).show(10, False)

sbersih.write.mode("overwrite").partitionBy("Borough") \
    .parquet("/content/lapisan_terkurasi/trips_spark")

sbersih.explain()  # cari BroadcastHashJoin, bukan SortMergeJoin
spark.stop()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 5.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 13.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.
[spark: hitung baris bersih] 12.25 s
+-------------+-------+
|Borough      |count  |
+-------------+-------+
|Manhattan    |2659609|
|Queens       |270315 |
|Unknown      |37609  |
|Brooklyn     |15724  |
|Bronx        |3074   |
|NaN          |340    |
|Staten Island|211    |
|EWR          |28     |
+-------------+-------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#245, trip_distance#247, PULocationID#7L, DOLocatio

Studi Kasus

In [ ]:
tabel_analitik = (
    bersih.assign(jam_mulai=bersih["tpep_pickup_datetime"].dt.floor("h"))
    .groupby(["borough_naik", "jam_mulai"], observed=True)
    .agg(
        jumlah_perjalanan=("total_amount", "size"),
        rata_tarif_per_mil=("tarif_per_mil", "median"),
        rata_kecepatan=("kecepatan_mph", "median"),
        suhu_c=("suhu_c", "first"),
        hujan=("hujan", "max"),
    )
    .reset_index()
)

# Ambang minimum: kelompok yang terlalu kecil tidak bisa disimpulkan
tabel_analitik = tabel_analitik[tabel_analitik["jumlah_perjalanan"] >= 30]

print(tabel_analitik
      .groupby(["borough_naik", "hujan"], observed=True)["rata_tarif_per_mil"]
      .median().unstack())

hujan               0        1
borough_naik                  
Brooklyn       6.9430   7.6550
Manhattan     10.5270  11.4000
Queens         5.4530   5.6365
Unknown        8.7255   9.8860


In [ ]:
PATH_ANALITIK = os.path.join(
    DIR_SIMPAN,
    "tabel_analitik_pricing.parquet"
)
PATH_KAMUS = os.path.join(
    DIR_SIMPAN,
    "kamus_data.md"
)
tabel_analitik.to_parquet(
    PATH_ANALITIK,
    index=False
)

isi_kamus = """# Kamus Data Tabel Analitik Pricing

Tabel memiliki tingkat rincian satu baris per borough per jam.

| Kolom | Satuan | Cara Penghitungan |
|---|---|---|
| borough_naik | - | Borough berdasarkan lokasi pickup perjalanan |
| jam_mulai | Waktu | Waktu pickup dibulatkan ke awal jam |
| jumlah_perjalanan | perjalanan | Jumlah perjalanan pada setiap borough dan jam |
| rata_tarif_per_mil | tarif/mil | Median tarif per mil pada setiap borough dan jam |
| rata_kecepatan | mph | Median kecepatan perjalanan pada setiap borough dan jam |
| suhu_c | °C | Suhu pada jam tersebut |
| hujan | mm/indikator | Nilai maksimum hujan pada jam tersebut |

## Aturan Pengolahan

- Data diagregasikan berdasarkan borough dan jam.
- Hanya kelompok dengan minimal 30 perjalanan yang digunakan.
- Tarif per mil dihitung menggunakan median.
- Kecepatan dihitung menggunakan median.
- Suhu menggunakan nilai pertama pada jam tersebut.
- Hujan menggunakan nilai maksimum pada jam tersebut.
"""

with open(PATH_KAMUS, "w", encoding="utf-8") as f:
    f.write(isi_kamus)

print("=== FILE BERHASIL DISIMPAN ===")
print("Parquet :", PATH_ANALITIK)
print("Kamus   :", PATH_KAMUS)

=== FILE BERHASIL DISIMPAN ===
Parquet : /content/drive/MyDrive/BigData/Praktikum3/tabel_analitik_pricing.parquet
Kamus   : /content/drive/MyDrive/BigData/Praktikum3/kamus_data.md


Latihan

Latihan 1 — Kecepatan Unduh dan Cache

Latihan ini menambahkan pengukuran kecepatan unduh dalam MB/detik dan membuktikan bahwa pemanggilan kedua menggunakan file dari cache sehingga tidak perlu mengunduh ulang.

In [ ]:
def unduh_aman_dengan_kecepatan(url, tujuan, percobaan=3):
    if os.path.exists(tujuan) and os.path.getsize(tujuan) > 1000:
        ukuran_mb = os.path.getsize(tujuan) / (1024**2)
        print(f"[CACHE] File sudah tersedia: {tujuan}")
        print(f"Ukuran: {ukuran_mb:.2f} MB")
        return

    print("[UNDUH] Mengunduh file...")

    awal = time.perf_counter()
    unduh_aman(url, tujuan, percobaan)
    durasi = time.perf_counter() - awal

    ukuran_mb = os.path.getsize(tujuan) / (1024**2)
    kecepatan = ukuran_mb / durasi

    print(f"Ukuran: {ukuran_mb:.2f} MB")
    print(f"Waktu: {durasi:.2f} detik")
    print(f"Kecepatan unduh: {kecepatan:.2f} MB/detik")

In [ ]:
PATH_LATIHAN = os.path.join(DIR_MENTAH, "yellow_tripdata_latihan.parquet")

unduh_aman_dengan_kecepatan(
    URL_TRIP,
    PATH_LATIHAN
)

[CACHE] File sudah tersedia: /content/lapisan_mentah/yellow_tripdata_latihan.parquet
Ukuran: 45.46 MB


Latihan 2 — Aturan Kualitas Tambahan

Latihan ini menambahkan pemeriksaan bahwa tip_amount tidak negatif dan total_amount konsisten dengan fare_amount serta tip_amount.

In [ ]:
aturan_tambahan = {
    "tip_amount >= 0": trip["tip_amount"] >= 0,

    "total >= fare + tip": (
        trip["total_amount"] >=
        trip["fare_amount"] + trip["tip_amount"]
    )
}

print("Hasil aturan tambahan:")

for nama, mask in aturan_tambahan.items():
    gagal = (~mask).sum()
    persen = 100 * gagal / len(trip)

    print(
        f"{nama}: "
        f"gagal {gagal:,} baris "
        f"({persen:.3f}%)"
    )

Hasil aturan tambahan:
tip_amount >= 0: gagal 225 baris (0.007%)
total >= fare + tip: gagal 25,160 baris (0.820%)


Latihan 3 — Join Lokasi Tujuan

Latihan ini menggabungkan lokasi tujuan berdasarkan DOLocationID dan menghitung sepuluh pasangan borough asal–tujuan dengan jumlah perjalanan terbanyak.

In [ ]:
tujuan = zona[["LocationID", "Borough", "Zone"]].rename(
    columns={
        "LocationID": "DOLocationID",
        "Borough": "borough_tujuan",
        "Zone": "zona_tujuan"
    }
)

asal_tujuan = bersih.merge(
    tujuan,
    on="DOLocationID",
    how="left",
    validate="many_to_one"
)

pasangan_borough = (
    asal_tujuan
    .groupby(
        ["borough_naik", "borough_tujuan"],
        observed=True
    )
    .size()
    .reset_index(name="jumlah_perjalanan")
    .sort_values("jumlah_perjalanan", ascending=False)
)

print("10 pasangan borough asal-tujuan paling sibuk:")
print(pasangan_borough.head(10).to_string(index=False))

10 pasangan borough asal-tujuan paling sibuk:
borough_naik borough_tujuan  jumlah_perjalanan
   Manhattan      Manhattan            2491566
      Queens      Manhattan             155261
   Manhattan         Queens              82354
   Manhattan       Brooklyn              62986
      Queens         Queens              59146
      Queens       Brooklyn              42187
     Unknown      Manhattan              18432
     Unknown        Unknown              13562
   Manhattan          Bronx               8865
    Brooklyn       Brooklyn               7910


Latihan 4 — Perbandingan Scaling

Latihan ini membandingkan rentang hasil StandardScaler dan MinMaxScaler serta melihat pengaruh scaling terhadap algoritma yang sensitif terhadap skala fitur.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

kolom_skala = ["trip_distance", "total_amount"]

# StandardScaler
standard = StandardScaler()
hasil_standard = standard.fit_transform(
    bersih[kolom_skala]
)

# MinMaxScaler
minmax = MinMaxScaler()
hasil_minmax = minmax.fit_transform(
    bersih[kolom_skala]
)

print("Rentang StandardScaler:")
print(
    "min =", hasil_standard.min(axis=0),
    "\nmax =", hasil_standard.max(axis=0)
)

print("\nRentang MinMaxScaler:")
print(
    "min =", hasil_minmax.min(axis=0),
    "\nmax =", hasil_minmax.max(axis=0)
)

Rentang StandardScaler:
min = [-0.77693567 -1.24472117] 
max = [21.21698061 27.75942439]

Rentang MinMaxScaler:
min = [0. 0.] 
max = [1. 1.]


Latihan 5 — Validasi many_to_one

Latihan ini membuktikan bahwa validate="many_to_one" akan menghasilkan error ketika kunci pada tabel zona tidak unik.

In [ ]:
zona_uji = pd.concat(
    [zona, zona.iloc[[0]]],
    ignore_index=True
)

print("Jumlah baris zona asli :", len(zona))
print("Jumlah baris zona uji  :", len(zona_uji))

try:
    hasil_uji = bersih.merge(
        zona_uji[["LocationID", "Borough", "Zone"]],
        left_on="PULocationID",
        right_on="LocationID",
        how="left",
        validate="many_to_one"
    )

except Exception as e:
    print("\nERROR YANG MUNCUL:")
    print(type(e).__name__)
    print(e)

Jumlah baris zona asli : 265
Jumlah baris zona uji  : 266

ERROR YANG MUNCUL:
MergeError
Merge keys are not unique in right dataset; not a many-to-one merge


Latihan 6 — Pipeline Inkremental

Latihan ini membuat pipeline yang hanya memproses bulan yang belum memiliki file keluaran sehingga pemanggilan kedua dapat dilewati dan tidak menghasilkan berkas baru.

In [ ]:
def pipeline_inkremental(bulan):
    """
    bulan: format YYYY-MM
    """

    nama_file = f"trips_bersih_{bulan}.parquet"
    path_output = os.path.join(DIR_KURASI, nama_file)

    if os.path.exists(path_output):
        print(f"[SKIP] {bulan} sudah tersedia.")
        print("Tidak ada berkas baru yang dibuat.")
        return None

    print(f"[PROSES] Memproses bulan {bulan}...")

    if bulan == "2023-01":
        hasil_bulan = pipeline(
            PATH_TRIP,
            URL_ZONA,
            cuaca,
            tarif_referensi
        )

        hasil_bulan.to_parquet(
            path_output,
            index=False
        )

        print(f"[SELESAI] File dibuat: {path_output}")
        return hasil_bulan

    else:
        print(f"[INFO] Data untuk {bulan} belum tersedia.")
        return None

In [ ]:
bulan_uji = "2023-01"

print("PEMANGGILAN PERTAMA")
pipeline_inkremental(bulan_uji)

print("\nPEMANGGILAN KEDUA")
pipeline_inkremental(bulan_uji)

PEMANGGILAN PERTAMA
[SKIP] 2023-01 sudah tersedia.
Tidak ada berkas baru yang dibuat.

PEMANGGILAN KEDUA
[SKIP] 2023-01 sudah tersedia.
Tidak ada berkas baru yang dibuat.
